In [1]:
from tqdm import tqdm
import pandas as pd

In [2]:
ground_truth_df = pd.read_csv("data/ground_truth.csv")
ground_truth_df = ground_truth_df.drop(labels="Unnamed: 0", axis="columns")

In [3]:
ground_truth_df.head()

,question,ground_truth_sql
0,How much have I spent in total across all my r...,SELECT SUM(amount) FROM expenses;
1,What is the total number of transactions I hav...,SELECT COUNT(*) FROM expenses;
2,Show me all the purchases I made in the 'Groce...,SELECT * FROM expenses WHERE category = 'Groce...
3,What is the total amount of money I spent on '...,SELECT SUM(amount) FROM expenses WHERE categor...
4,What is the most expensive single purchase I h...,SELECT MAX(amount) FROM expenses;


In [4]:
ground_truth = ground_truth_df.to_dict(orient="records")

In [5]:

from rag_evaluator import RAGEvaluator
from google import genai
from sqlite_db import get_db_connection

llm_client = genai.Client()
db_conn = get_db_connection()
evaluator = RAGEvaluator(llm_client=llm_client)

In [6]:
def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results

In [7]:
from concurrent.futures import ThreadPoolExecutor

In [8]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, evaluator.get_sql_results)

100%|██████████| 50/50 [00:31<00:00,  1.58it/s]


In [9]:
evaluator.evaluate(results)

In [10]:
evaluator.evaluation

{'valid_execution_rate': 1.0, 'execution_accuracy': 0.9}